In [9]:
import pandas as pd

df = pd.read_csv('transformed/parcels.csv')

CLEAN_PARCEL_COLS = [
    'parcel_predix',
    'parcel_street',
    'parcel_suffix',
    'parcel_unit',
]

# df.sample(5)

df.groupby('parcel_unit').size().sort_values().reset_index().sample(10)

/tmp/ipykernel_5562/3409847442.py:3: DtypeWarning: Columns (0: parcel) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('transformed/parcels.csv')


,parcel_unit,0
27,D-14,1
1328,34,3
1132,422,1
797,144,1
1464,1004,5
237,872,1
30,D-1261,1
732,I-7535,1
1566,402,19
803,1241,1


In [ ]:
def try_cast_to_int(string):
    try:
        return int(string.strip())
    except ValueError:
        return None
    
def adjust_shortened_address_range(start, end):
    start = str(start)
    end = str(end)

    # Handle formats like '10002-4'
    to_add = start[: -len(end)]
    end_adj = int(to_add + end)
    if end_adj > int(start):
        return end_adj
    
    # If we couldn't find an adjusted endpoint that is
    # greater than the start point, give up and return
    # the starting point
    return int(start)

def addr_to_min_max(string):
    SEPARATORS = ['-', '&', 'to']

    # Handle addresses like '100-102'
    string = str(string).lower()
    for sep in SEPARATORS:
        if string.count(sep) == 1:
            start, end = string.split(sep)

            start = try_cast_to_int(start)
            end = try_cast_to_int(end)

            if start and end and end < start:
                return start, adjust_shortened_address_range(start, end)

            return start, end

    # Likely only a single number or an invalid number.
    integer = try_cast_to_int(string)
    return integer, integer

def addr_to_min(string):
    addr_min, _ = addr_to_min_max(string)
    return addr_min

def addr_to_max(string):
    _, addr_max = addr_to_min_max(string)
    return addr_max

df['parcel_street'] = df['parcel_street'].apply(lambda x: x.strip() if not pd.isna(x) else None)
df['parcel_addr_min'] = df['parcel_addr'].apply(lambda x: addr_to_min(x))
df['parcel_addr_max'] = df['parcel_addr'].apply(lambda x: addr_to_max(x))

,parcel_addr,parcel_predir,parcel_street,parcel_addr_min,parcel_addr_max,range
27741,3524,W,50,3524.0,3524.0,0.0
73667,1563,E,41,1563.0,1563.0,0.0
74760,2816-2828,NaN,EUCLID,2816.0,2828.0,12.0
27522,3412,W,45,3412.0,3412.0,0.0
119809,7614,NaN,HOUGH,7614.0,7614.0,0.0
146848,16709,NaN,DEFOREST,16709.0,16709.0,0.0
56772,4583,W,130,4583.0,4583.0,0.0
114016,1454,E,116,1454.0,1454.0,0.0
84199,833,E,156,833.0,833.0,0.0
119199,2325,NaN,BELVOIR,2325.0,2325.0,0.0


In [70]:
df['parcel_street'] = df['parcel_street'].apply(lambda x: x.strip().upper() if not pd.isna(x) else None)
df.groupby('parcel_street').size().reset_index().rename(columns={0: 'count'}).sample(10)

,parcel_street,count
1240,HAZELMERE,17
217,75 TO 90,1
1684,MEADOWBROOK,67
1758,MORRIS,5
454,BLATT,6
355,AUBURN,73
329,ARDOON,25
4,103,609
1797,NAPLES,51
158,4,98


In [61]:
df[df.parcel_street == '143']

,parcelpin,par_addr_all,parcel_addr,parcel_predir,parcel_street,parcel_suffix,parcel_unit,parcel_city,parcel_zip,parcel_owner,...,leadSafeCertificateActiveFlag,transfers_in_5y,myPlaceLink,Survey2022_Grade,Survey2022_PhotoLink,taxDelinquencyAmount,max_age,min_age,min_com_age,max_com_age
49270,02318040,"0 W 143 ST, CLEVELAND, OH, 44135",NaN,W,143,ST,NaN,CLEVELAND,44135.0,CITY OF CLEVE,...,0,0,https://myplace.cuyahogacounty.gov/MDIzMTgwNDA...,N/A,https://wdwot.s3.amazonaws.com/blextoid/637cdb...,0.00,0.0,0.0,0.0,0.0
49271,02318039,"0 W 143 ST, CLEVELAND, OH, 44135",NaN,W,143,ST,NaN,CLEVELAND,44135.0,CITY OF CLEVE,...,0,0,https://myplace.cuyahogacounty.gov/MDIzMTgwMzk...,N/A,https://wdwot.s3.amazonaws.com/blextoid/637cdb...,0.00,0.0,0.0,0.0,0.0
49272,02318038,"0 W 143 ST, CLEVELAND, OH, 44135",NaN,W,143,ST,NaN,CLEVELAND,44135.0,CITY OF CLEVE,...,0,0,https://myplace.cuyahogacounty.gov/MDIzMTgwMzg...,N/A,https://wdwot.s3.amazonaws.com/blextoid/637cdb...,0.00,0.0,0.0,0.0,0.0
49273,02318037,"0 W 143 ST, CLEVELAND, OH, 44135",NaN,W,143,ST,NaN,CLEVELAND,44135.0,CITY OF CLEVE,...,0,0,https://myplace.cuyahogacounty.gov/MDIzMTgwMzc...,N/A,https://wdwot.s3.amazonaws.com/blextoid/637cdb...,0.00,0.0,0.0,0.0,0.0
49274,02318036,"0 W 143 ST, CLEVELAND, OH, 44135",NaN,W,143,ST,NaN,CLEVELAND,44135.0,CITY OF CLEVE,...,0,0,https://myplace.cuyahogacounty.gov/MDIzMTgwMzY...,N/A,https://wdwot.s3.amazonaws.com/blextoid/637cdb...,0.00,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154665,13820121,"4364 E 143 ST, CLEVELAND, OH, 44128",4364,E,143,ST,NaN,CLEVELAND,44128.0,"JIAO'S HOLDINGS, LLC",...,0,4,https://myplace.cuyahogacounty.gov/MTM4MjAxMjE...,C,https://wdwot.s3.amazonaws.com/blextoid/640613...,0.00,1950.0,1950.0,0.0,0.0
154666,13820120,"4368 E 143 ST, CLEVELAND, OH, 44128",4368,E,143,ST,NaN,CLEVELAND,44128.0,"MBM CAPITAL HOLDINGS, LLC",...,1,2,https://myplace.cuyahogacounty.gov/MTM4MjAxMjA...,C,https://wdwot.s3.amazonaws.com/blextoid/640612...,0.00,1950.0,1950.0,0.0,0.0
154667,13820119,"4372 E 143 ST, CLEVELAND, OH, 44128",4372,E,143,ST,NaN,CLEVELAND,44128.0,TMDW PROPERTIES LLC,...,0,1,https://myplace.cuyahogacounty.gov/MTM4MjAxMTk...,C,https://wdwot.s3.amazonaws.com/blextoid/640612...,2173.91,1950.0,1950.0,0.0,0.0
154668,13820118,"4376 E 143 ST, CLEVELAND, OH, 44128",4376,E,143,ST,NaN,CLEVELAND,44128.0,IDK 2024 LLC,...,0,2,https://myplace.cuyahogacounty.gov/MTM4MjAxMTg...,B,https://wdwot.s3.amazonaws.com/blextoid/640611...,0.00,1949.0,1949.0,0.0,0.0


In [ ]:
t['len'] = t['parcel_addr'].apply(lambda x: len(x) if not pd.isna(x) else 0)
t = t.sort_values('len')
t = t[['parcel_addr', 'parcel_addr_min', 'parcel_addr_max']]
t.to_csv('test.csv', index=False)

In [20]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
!pip install pyarrow

   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
    --------------------------------------- 0.5/27.4 MB 3.3 MB/s eta 0:00:09
   - -------------------------------------- 1.3/27.4 MB 3.2 MB/s eta 0:00:09
   --- ------------------------------------ 2.1/27.4 MB 3.6 MB/s eta 0:00:08
   --- ------------------------------------ 2.6/27.4 MB 3.4 MB/s eta 0:00:08
   ---- ----------------------------------- 3.4/27.4 MB 3.3 MB/s eta 0:00:08
   ------ --------------------------------- 4.2/27.4 MB 3.4 MB/s eta 0:00:07
   ------- -------------------------------- 5.2/27.4 MB 3.5 MB/s eta 0:00:07
   -------- ------------------------------- 6.0/27.4 MB 3.6 MB/s eta 0:00:06
   ---------- ----------------------------- 7.1/27.4 MB 3.7 MB/s eta 0:00:06
   ----------- ---------------------------- 7.9/27.4 MB 3.8 MB/s eta 0:00:06
   ------------- -------------------------- 8.9/27.4 MB 3.9 MB/s eta 0:00:05
   -------------- ------------------------- 10.0/27.4 MB 4.0 MB/s eta 0:00:05
   --


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import os
import pandas as pd 
import json 
from dotenv import load_dotenv
# for file in os.listdir('transformed'):
#     pd.read_csv(f"transformed/{file}").sample(600).to_csv(f"temp/{file}", index=False)

In [23]:
load_dotenv()

key = os.environ.get('SERVICE_ROLE_KEY')

In [24]:
key

'sb_secret_J4Lkzt6xzaG7u2-AxMU5Jw_wDkN_R7q'

In [25]:
import requests

In [26]:
url = 'https://kvzikdxyxvziagomfvii.supabase.co/rest/v1/civil_tickets?select=*',
HEADERS = {
    'apikey': key,
    'Authorization': f'Bearer {key}',
    'Content-Type': 'application/json',
}

r = requests.get(url, headers=HEADERS)
r.raise_for_status()

InvalidSchema: No connection adapters were found for "('https://kvzikdxyxvziagomfvii.supabase.co/rest/v1/civil_tickets?select=*',)"

In [ ]:
{'parcel': '02623074', 'par_addr_all': '18313 FAIRWAY AVE, CLEVELAND, OH, 44135', 'parcel_owner': 'COLLINS, KATHRYNE E.', 'std_deeded_owner': 'COLLINS, KATHRYNE E.', 'grantor': 'FRIEDL, SANDRA J.', 'grantee': 'COLLINS, KATHRYNE E.', 'last_transfer_date': 1571867100000, 'tax_assessed_total': 57960.0, 'activerentalregistrationflag': 0, 'activecertificateapprovingrentaloccupancyflag': 0, 'leadsafecertificateactiveflag': 0, 'transfers_in_5y': 0, 'myplacelink': 'https://myplace.cuyahogacounty.gov/MDI2MjMwNzQ=?city=OTk=&searchBy=UGFyY2Vs&dataRequested=UHJvcGVydHkgU3VtbWFyeSBSZXBvcnQ=', 'survey2022_grade': 'A', 'survey2022_photolink': 'https://wdwot.s3.amazonaws.com/blextoid/63ed38c8adf0fbba52af05e9/697ce05977afd1ab.jpg?t=1676490951', 'taxdelinquencyamount': 0.0, 'max_age': 1959.0, 'min_age': 1959.0, 'min_com_age': 0.0, 'max_com_age': 0.0, 'owner_clean': 'COLLINS KATHRYNE E ', 'civil_tickets': 0.0, 'complaints_health': 0.0, 'code_violations': 0.0, 'complaints_311': 0.0}]

In [45]:
pd.set_option('display.max_columns', None)

In [78]:
h = pd.read_csv('transformed/complaints_health.csv')
p = pd.read_csv('transformed/parcels.csv')


C:\Users\jackv\AppData\Local\Temp\ipykernel_1352\1428274101.py:1: DtypeWarning: Columns (0: farm_animal, 1: odor_type, 2: parcel) have mixed types. Specify dtype option on import or set low_memory=False.
  h = pd.read_csv('transformed/complaints_health.csv')


In [79]:
h.merge(p[['parcel']], on='parcel', how='left_anti')

,objectid,id,complaint_number,submit_date,complaint_type,complaint_input,complaint_inspector,complaint_status,complaint_outcome,food_complaint,farm_animal,insect_vermin,odor_type,odor_strength,problem_location_name,parcel
6,8,12759,2021012759,1636675200000,Accumulation of Garbage,Sharia Clark,Tina King,Resolved,Violation Notice Issued,NaN,NaN,NaN,NaN,NaN,NaN,00109029
31,35,12787,2021012787,1636934400000,Accumulation of Garbage,Robert Peterson,Charlotte Ford,Resolved,Ticketed,NaN,NaN,NaN,NaN,NaN,NaN,12715015
42,51,12803,2021012803,1636934400000,Other,Sharia Clark,Robert Peterson,Resolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00117006
46,55,12807,2021012807,1636934400000,Accumulation of Garbage,Robert Peterson,Charlotte Ford,Resolved,Orange Tag,NaN,NaN,NaN,NaN,NaN,NaN,12319041
48,57,12809,2021012809,1636934400000,Accumulation of Garbage,Robert Peterson,Charlotte Ford,Resolved,Violation Notice Issued,NaN,NaN,NaN,NaN,NaN,NaN,12715090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32868,36804,39894,2026039894,1778198400000,Accumulation of Garbage,Ward-12,Jessica Jennings,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1805139
32869,36806,39896,2026039896,1778284800000,High Grass/Weeds,Website Intake,Anthony Gober,Active,NaN,NaN,NaN,NaN,NaN,NaN,Neighborâ€™s Home,1509147
32870,36808,39898,2026039898,1778371200000,Accumulation of Garbage,Website Intake,Anthony Gober,Active,NaN,NaN,NaN,NaN,NaN,NaN,Residential Home,1321121
32871,36809,39899,2026039899,1778371200000,Food Complaints,Website Intake,NaN,New,NaN,Restaurant/Store Conditions,NaN,NaN,NaN,NaN,Rebel (marathon),11425008


In [77]:
def filter_to_known_parcels(df, parcel_df):
    return df.merge(parcel_df[['parcel']].drop_duplicates(), on='parcel').copy()


print('health complaints')

complaints_health = pd.read_json('source/complaints_health.json')
complaints_health.columns = [x.lower().strip() for x in complaints_health.columns]

complaints_health = complaints_health.rename(columns={'dw_parcel': 'parcel'})

complaints_health = filter_to_known_parcels(complaints_health, p)

complaints_health[[
    'objectid',
    'id',
    'complaint_number',
    'submit_date',
    'complaint_type',
    'complaint_input',
    'complaint_inspector',
    'complaint_status',
    'complaint_outcome',
    'food_complaint',
    'farm_animal',
    'insect_vermin',
    'odor_type',
    'odor_strength',
    'problem_location_name',
    'parcel',
]]

complaints_health.merge(p[['parcel']], on='parcel', how='left_anti')

# .to_csv('transformed/complaints_health.csv', index=False)


health complaints


,objectid,id,complaint_number,submit_date,submit_time,complaint_type,complaint_input,complaint_inspector,complaint_status,complaint_outcome,food_complaint,farm_animal,insect_vermin,odor_type,odor_strength,problem_location_name,problem_address,problem_street_direction,problem_street_name,problem_street_type,problem_unit_number,problem_city,problem_zip_code,problem_date_time,mac_complaint_id,permanent_parcel_number,census_tract,ward_number,submit_datetime,dw_ward,dw_ward_2014,dw_ward_2026,dw_neighborhood,dw_census_tract,parcel


In [72]:
df.groupby(['ObjectId']).size().sort_values()

ObjectId
1        1
2        1
3        1
4        1
5        1
        ..
36778    1
36779    1
36780    1
36781    1
36782    1
Length: 36782, dtype: int64

In [36]:
json.loads(df.to_json(orient='records'))[0]

{'parcel': '02630042',
 'par_addr_all': '19338 HIPPLE AVE, CLEVELAND, OH, 44135',
 'parcel_owner': 'KOMPAN, JESSICA MARIE AND KOMPAN, DIANE',
 'std_deeded_owner': 'KOMPAN, JESSICA MARIE AND KOMPAN, DIANE',
 'grantor': 'RIVERA, ELENA J.',
 'grantee': 'KOMPAN, JESSICA MARIE AND KOMPAN, DIANE',
 'last_transfer_date': 1622655900000.0,
 'tax_assessed_total': 57410.0,
 'activerentalregistrationflag': 0,
 'activecertificateapprovingrentaloccupancyflag': 0,
 'leadsafecertificateactiveflag': 0,
 'transfers_in_5y': 1,
 'myplacelink': 'https://myplace.cuyahogacounty.gov/MDI2MzAwNDI=?city=OTk=&searchBy=UGFyY2Vs&dataRequested=UHJvcGVydHkgU3VtbWFyeSBSZXBvcnQ=',
 'survey2022_grade': 'B',
 'survey2022_photolink': 'https://wdwot.s3.amazonaws.com/blextoid/6369639fc1adb4de04ce8b57/ef5a954543e71c90.jpg?t=1667851165',
 'taxdelinquencyamount': 0.0,
 'max_age': 1964.0,
 'min_age': 1964.0,
 'min_com_age': 0.0,
 'max_com_age': 0.0,
 'owner_clean': 'KOMPAN JESSICA MARIE AND KOMPAN DIANE',
 'civil_tickets': 0.0,

In [18]:
len(df)

161327